## Assign MATPOWER properties to geoJSON file features

In [2]:
import json

In [3]:
# Load the GeoJSON and MATPOWER file
with open('./IEEE_30_bus_system.geojson', 'r') as file:
    gdf = json.load(file)

from IEEE_30_bus_system import IEEE_30_bus_system
ppc = IEEE_30_bus_system()

In [7]:
bus_type = {
    1: 'PQ bus',
    2: 'PV bus',
    3: 'Reference bus',
    4: 'Isolated bus'
}
bus_keys = ["bus_i", "type", "Pd", "Qd", "Gs", "Bs", "area", "Vm", "Va", "baseKV", "zone", "Vmax", "Vmin"]
gen_keys = ["bus", "Pg", "Qg", "Qmax", "Qmin", "Vg", "mBase", "status", "Pmax", "Pmin", "Pc1", "Pc2", "Qc1min", "Qc1max", "Qc2min", "Qc2max", "ramp_agc", "ramp_10", "ramp_30", "ramp_q", "apf"]
branch_keys = ["fbus", "tbus", "r", "x", "b", "rateA", "rateB", "rateC", "ratio", "angle", "status", "angmin", "angmax"]

def find_properties_point(bus_id, ppc):
    properties = {}
    
    # Check in buses
    for bus in ppc["bus"]:
        if bus[0] == bus_id:
            for i, key in enumerate(bus_keys):
                if i > 1:
                    properties[key] = bus[i]
            properties["bus_type"] = bus_type[bus[1]]
            break

    # Check in generators
    for i, gen in enumerate(ppc["gen"]):
        if gen[0] == bus_id:
            for j, key in enumerate(gen_keys):
                if j > 0:
                    properties[key] = gen[j]
            properties["has_gen"] = 1
            properties["gen_index"] = i
            break

    # Check in branches
    connected_branches = []
    for i, branch in enumerate(ppc["branch"]):
        if branch[0] == bus_id or branch[1] == bus_id:
            connected_branches.append(i+1)
    properties["connected_branches"] = connected_branches

    return properties

def find_properties_branch(branch_id, ppc):
    properties = {}
    for i, key in enumerate(branch_keys):
        properties[key] = ppc["branch"][branch_id-1][i]

    return properties

# Iterate over GeoDataFrame
for idx, row in enumerate(gdf['features']):
    if row["geometry"]["type"] == "Point":
        properties = find_properties_point(row["properties"]["bus"], ppc)
    else:
        properties = find_properties_branch(row["properties"]["branch"], ppc)
    
    for k, v in properties.items():
        gdf['features'][idx]['properties'][k] = v


41 1 {'Pd': 0.0, 'Qd': 0.0, 'Gs': 0.0, 'Bs': 0.0, 'area': 1.0, 'Vm': 1.0, 'Va': 0.0, 'baseKV': 135.0, 'zone': 1.0, 'Vmax': 1.05, 'Vmin': 0.95, 'bus_type': 'Reference bus', 'Pg': 23.54, 'Qg': 0.0, 'Qmax': 150.0, 'Qmin': -20.0, 'Vg': 1.0, 'mBase': 100.0, 'status': 1.0, 'Pmax': 80.0, 'Pmin': 0.0, 'Pc1': 0.0, 'Pc2': 0.0, 'Qc1min': 0.0, 'Qc1max': 0.0, 'Qc2min': 0.0, 'Qc2max': 0.0, 'ramp_agc': 0.0, 'ramp_10': 0.0, 'ramp_30': 0.0, 'ramp_q': 0.0, 'apf': 0.0, 'has_gen': 1, 'gen_index': 0, 'connected_branches': [1, 2]}
42 2 {'Pd': 21.7, 'Qd': 12.7, 'Gs': 0.0, 'Bs': 0.0, 'area': 1.0, 'Vm': 1.0, 'Va': 0.0, 'baseKV': 135.0, 'zone': 1.0, 'Vmax': 1.1, 'Vmin': 0.95, 'bus_type': 'PV bus', 'Pg': 60.97, 'Qg': 0.0, 'Qmax': 60.0, 'Qmin': -20.0, 'Vg': 1.0, 'mBase': 100.0, 'status': 1.0, 'Pmax': 80.0, 'Pmin': 0.0, 'Pc1': 0.0, 'Pc2': 0.0, 'Qc1min': 0.0, 'Qc1max': 0.0, 'Qc2min': 0.0, 'Qc2max': 0.0, 'ramp_agc': 0.0, 'ramp_10': 0.0, 'ramp_30': 0.0, 'ramp_q': 0.0, 'apf': 0.0, 'has_gen': 1, 'gen_index': 1, 'connec

In [ ]:
# Use json.dump to write the dictionary to a file
with open('output.geojson', 'w') as f:
    json.dump(gdf, f)